<a href="https://colab.research.google.com/github/ZahranAzariaAnvaya/BigData26_A_2411531005_ZahranAzariaAnvaya/blob/main/Praktikum2/BD_A_P02_2411531005_ZahranAzariaAnvaya.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Install Faker**

In [1]:
!pip install faker -q

# **Praktikum**

## **K1. Import Library**

Menyiapkan seluruh pustaka yang dipakai sepanjang praktikum, seperti numpy untuk operasi numerik/acak, pandas untuk manipulasi tabel, Faker untuk membangkitkan data palsu (nama, kota, dll), dan random untuk memilih variasi format secara acak.

In [2]:
import numpy as np
import pandas as pd
from faker import Faker
import random

### **Menghubungkan Google Drive & Menyiapkan Struktur Folder**

In [3]:
from google.colab import drive
drive.mount("/content/drive")

import os
DIR_KERJA  = "/content/data"                          # sementara, cepat
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum2"  # permanen
os.makedirs(DIR_KERJA, exist_ok=True)
os.makedirs(DIR_SIMPAN, exist_ok=True)
print(os.listdir(DIR_SIMPAN))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['transaksi_mentah.csv', 'transaksi_bersih.csv', 'transaksi_bersih_seed7.csv', 'transaksi_mentah_seed7.csv']


## **K.2 Membuat Dataset Sintetis**

Sel ini mensimulasikan proses acquisition data transaksi marketplace. SEED = 42 dipanggil untuk numpy, random, dan Faker sekaligus agar seluruh proses acak bisa direproduksi identik oleh semua mahasiswa. Data sengaja "dikotori" denganvariasi format harga (angka polos, "Rp", desimal, spasi), variasi format tanggal (ISO, DD/MM/YYYY, DD-MM-YYYY), variasi kapitalisasi, missing value pada 3 kolom, serta 15 baris duplikat agar meniru kondisi data mentah dunia nyata.

In [4]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    # Variasi format harga: angka polos, ada "Rp", ada desimal ".0", ada spasi
    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar} ",
    ]
    harga = random.choice(harga_variants)

    # Variasi format tanggal: ISO, DD/MM/YYYY, DD-MM-YYYY
    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)
    if random.random() < 0.3:
        metode = metode.lower()
    if random.random() < 0.2:
        kategori = kategori.upper() + "  "

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None])  # rating opsional

    rows.append({
        "transaction_id": trx_id, "customer_name": nama_pelanggan, "product_name": produk.strip(),
        "category": kategori, "price": harga, "quantity": qty, "payment_method": metode,
        "transaction_date": tanggal, "shipping_city": kota, "rating": rating,
    })

df = pd.DataFrame(rows)

# Suntikkan missing value pada beberapa kolom
for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx = df.sample(frac=frac, random_state=SEED).index
    df.loc[idx, col] = np.nan

# Duplikasi 15 baris (mensimulasikan transaksi yang tercatat dua kali)
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv("transaksi_mentah.csv", index=False)
print("Jumlah baris:", len(df))

Jumlah baris: 515


In [5]:
df.head()

,transaction_id,customer_name,product_name,category,price,quantity,payment_method,transaction_date,shipping_city,rating
0,TRX00305,Ophelia Hartati,Quo Basic,Buku,50000,4,kartu kredit,2026-07-14,Blitar,1.0
1,TRX00500,Cut Maya Wijayanti,Delectus,Elektronik,Rp500.000,1,E-Wallet,2026-07-10,Lubuklinggau,NaN
2,TRX00442,"Dr. Ridwan Utama, M.Pd",Animi Max,Elektronik,25000,1,COD,2026-08-26,Probolinggo,2.0
3,TRX00154,"Viman Suwarno, S.H.",Consectetur,Fashion,Rp250.000,1,COD,28/07/2026,Tual,4.0
4,TRX00074,NaN,Eius,Olahraga,250000.0,4,NaN,02/07/2026,NaN,1.0


## **K-3. Deteksi dan Penanganan Missing Value**

Menghitung jumlah nilai kosong (NaN) per kolom, sebagai dasar untuk menentukan strategi penanganan per kolom pada sel berikutnya. Berdasarkan modul: customer_name ≈ 20 kosong, payment_method ≈ 16 kosong, shipping_city ≈ 30 kosong, rating ≈ 166 kosong.

In [6]:
print(df.isnull().sum())

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              166
dtype: int64


In [7]:
df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")
print("Jumlah baris setelah dropma():", len(df))

Jumlah baris setelah dropma(): 495


In [8]:
# Buktikan customer_name dan payment_method sudah tidak ada yang kosong (karena dihapus)
print("Cek missing value setelah dropna():")
print(df.isnull().sum())

print("\nJumlah baris tersisa:", len(df))

Cek missing value setelah dropna():
transaction_id        0
customer_name         0
product_name          0
category              0
price                 0
quantity              0
payment_method        0
transaction_date      0
shipping_city         0
rating              158
dtype: int64

Jumlah baris tersisa: 495


In [9]:
# Tampilkan baris yang shipping_city-nya tadinya kosong, sekarang jadi "Tidak Diketahui"
print(df[df["shipping_city"] == "Tidak Diketahui"][["transaction_id", "shipping_city"]].head(10))
print("\nJumlah baris dengan shipping_city = 'Tidak Diketahui':", (df["shipping_city"] == "Tidak Diketahui").sum())


    transaction_id    shipping_city
7         TRX00010  Tidak Diketahui
13        TRX00085  Tidak Diketahui
33        TRX00010  Tidak Diketahui
61        TRX00195  Tidak Diketahui
102       TRX00195  Tidak Diketahui
131       TRX00085  Tidak Diketahui
252       TRX00372  Tidak Diketahui
272       TRX00407  Tidak Diketahui
471       TRX00407  Tidak Diketahui
507       TRX00372  Tidak Diketahui

Jumlah baris dengan shipping_city = 'Tidak Diketahui': 10


## **K-4. Deteksi dan Penanganan Duplicate**

df.duplicated() mendeteksi baris yang seluruh kolomnya identik dengan baris lain. drop_duplicates() membuang baris duplikat tersebut, menyisakan satu salinan per transaksi.

In [10]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df['transaction_id'].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490


## **K-5. Koreksi Tipe Data dan Standardisasi Format**

### **a. Standardisasi teks kategorikal**

.astype("string") (bukan .astype(str)) dipakai agar NaN tetap dikenali sebagai missing, bukan berubah jadi teks "nan". .str.strip().str.title() merapikan spasi berlebih dan menyeragamkan kapitalisasi. Khusus "COD" yang merupakan singkatan, dikembalikan ke huruf kapital penuh karena Title Case akan salah mengubahnya menjadi "Cod".

In [11]:
for col in ["category", "payment_method", "shipping_city"]:
    df[col] = df[col].astype("string").str.strip().str.title()

# "Cod" adalah singkatan; kembalikan ke huruf kapital penuh setelah Title Case
df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

### **b. Koreksi tipe data pada kolom price (dari teks bercampur simbol, menjadi numerik)**

Kolom price awalnya bertipe teks karena bercampur simbol ("Rp", titik ribuan, spasi). Fungsi ini membersihkan simbol-simbol tersebut lalu mengonversinya menjadi float; jika konversi gagal, nilainya diisi NaN.

In [12]:
def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
    try:
        return float(x)
    except ValueError:
        return np.nan

df["price"] = df["price"].apply(bersihkan_harga)

### **c. Standardisasi format tanggal ke YYYY-MM-DD**

Kolom tanggal tercampur 3 format (ISO, DD/MM/YYYY, DD-MM-YYYY). Fungsi mencoba format eksplisit satu per satu untuk tiap nilai, bukan memakai pd.to_datetime(..., format="mixed", dayfirst=True) — kombinasi itu berbahaya karena bisa ikut membalik tanggal ISO yang sebenarnya sudah tidak ambigu (mis. 2026-07-11 salah terbaca jadi 2026-11-07).

In [13]:
def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")

### **d. Finalisasi tipe data**

Mengunci tipe data akhir: quantity sebagai integer dan price sebagai float, memastikan kedua kolom numerik siap dipakai untuk perhitungan/analisis.

In [14]:
df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)

## **K-6. Ekspor Dataset Bersih**

Menyimpan hasil akhir ke transaksi_bersih.csv (nama file harus persis seperti ini karena akan dipakai ulang di Praktikum 3 dan dikonversi ke transaksi_bersih.parquet).

In [15]:
df.to_csv("transaksi_bersih.csv", index=False)
print("Dataset bersih tersimpan:", len(df), "baris")

df.to_csv(f"{DIR_SIMPAN}/transaksi_bersih.csv", index=False)

Dataset bersih tersimpan: 490 baris


In [16]:
df.head()

,transaction_id,customer_name,product_name,category,price,quantity,payment_method,transaction_date,shipping_city,rating
0,TRX00305,Ophelia Hartati,Quo Basic,Buku,50000.0,4,Kartu Kredit,2026-07-14,Blitar,1.0
1,TRX00500,Cut Maya Wijayanti,Delectus,Elektronik,500000.0,1,E-Wallet,2026-07-10,Lubuklinggau,NaN
2,TRX00442,"Dr. Ridwan Utama, M.Pd",Animi Max,Elektronik,25000.0,1,COD,2026-08-26,Probolinggo,2.0
3,TRX00154,"Viman Suwarno, S.H.",Consectetur,Fashion,250000.0,1,COD,2026-07-28,Tual,4.0
5,TRX00132,Ir. Ilyas Setiawan,Ducimus Pro,Rumah Tangga,250000.0,4,COD,2026-09-16,Subulussalam,2.0


# **Studi Kasus**

**1. Mengapa angka tim IT (515) dan tim Finance (490) bisa berbeda?**

Kedua angka itu berasal dari tahap pipeline yang berbeda, bukan sumber data yang berbeda. Angka 515 adalah jumlah baris pada transaksi_mentah.csv, data mentah hasil acquisition, sebelum melalui proses pembersihan apa pun. Angka 490 adalah jumlah baris pada transaksi_bersih.csv, setelah melewati seluruh tahap preprocessing. 15 baris duplicate dihapus dan 10 baris dengan data wajib kosong (customer_name atau payment_method) dibuang, sehingga 515 − 15 − 10 = 490. Jadi tim IT kemungkinan melihat data mentah langsung dari sistem pencatatan, sedangkan tim Finance memakai data yang sudah difilter untuk analisis, ini bukan berarti salah satu tim keliru mencatat.

**2. Apakah 490 baris "lebih benar" dibanding 515 baris?**

Ya, dalam konteks analisis dan pengambilan keputusan bisnis, 490 lebih layak dipakai, ini dikaitkan langsung ke dimensi Veracity pada 5V. Veracity mengukur seberapa akurat dan bisa dipercaya suatu data. 515 mengandung baris yang tidak bisa dipercaya sebagai transaksi valid: 15 di antaranya adalah transaksi yang sama tercatat dua kali (duplicate), sehingga jika dihitung apa adanya akan menggandakan nilai penjualan. Sisanya, 10 baris kehilangan informasi wajib (nama pembeli atau metode pembayaran) sehingga tidak bisa diverifikasi sebagai transaksi yang sah. Volume yang besar (515) tidak ada gunanya kalau sebagian isinya tidak akurat, inilah prinsip "garbage in, garbage out". Jadi 490 bukan angka yang lebih kecil karena ada data hilang, tapi angka yang veracity-nya sudah divalidasi.

**3. Bagaimana menjelaskan kolom rating yang dibiarkan kosong ke tim Finance yang ingin tahu "rating rata-rata semua transaksi"?**

Jelaskan bahwa rating memang bersifat opsional,  sehingga pembeli tidak diwajibkan memberi rating setiap kali bertransaksi, sehingga nilai kosong (NaN) di kolom ini bukan kesalahan sistem, melainkan representasi valid dari "pembeli tidak memberi rating". Jika nilai kosong ini dipaksa diisi dengan angka tebakan (misalnya rata-rata keseluruhan atau nilai netral seperti 3), angka "rating rata-rata semua transaksi" yang dihasilkan justru akan bias dan menyesatkan, karena seolah-olah semua transaksi punya penilaian padahal tidak. Solusi yang benar adalah dengan menghitung rata-rata rating hanya dari transaksi yang benar-benar memiliki rating (df["rating"].mean() di pandas otomatis mengabaikan NaN), lalu sampaikan juga berapa persen transaksi yang punya rating (misalnya "rating rata-rata 3,2 dari 324 transaksi yang memberi rating, atau 66% dari total transaksi"). Dengan begitu tim Finance tahu angka tersebut mewakili sebagian data, bukan seluruh transaksi.

## **Latihan 1 - Ganti SEED jadi 7 dan bandingkan**

Jalankan ulang semua pipline

In [17]:
SEED7 = 7
np.random.seed(SEED7)
random.seed(SEED7)
fake7 = Faker("id_ID")
Faker.seed(SEED7)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows7 = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake7.name()
    produk = fake7.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar} ",
    ]
    harga = random.choice(harga_variants)

    tgl = fake7.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)
    if random.random() < 0.3:
        metode = metode.lower()
    if random.random() < 0.2:
        kategori = kategori.upper() + "  "

    kota = fake7.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None])

    rows7.append({
        "transaction_id": trx_id, "customer_name": nama_pelanggan, "product_name": produk.strip(),
        "category": kategori, "price": harga, "quantity": qty, "payment_method": metode,
        "transaction_date": tanggal, "shipping_city": kota, "rating": rating,
    })

df7 = pd.DataFrame(rows7)

for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx = df7.sample(frac=frac, random_state=SEED7).index
    df7.loc[idx, col] = np.nan

dup_rows7 = df7.sample(n=15, random_state=SEED7)
df7 = pd.concat([df7, dup_rows7], ignore_index=True)
df7 = df7.sample(frac=1, random_state=SEED7).reset_index(drop=True)

jumlah_mentah_7 = len(df7)
df7.to_csv(f"{DIR_SIMPAN}/transaksi_mentah_seed7.csv", index=False)
print("(SEED=7): Dataset Sintetis")
print("Jumlah baris:", jumlah_mentah_7)

# Deteksi & Penanganan Missing Value
print("\n\n == Missing Value ==")
print(df7.isnull().sum())

df7 = df7.dropna(subset=["customer_name", "payment_method"])
df7["shipping_city"] = df7["shipping_city"].fillna("Tidak Diketahui")
print("\nJumlah baris setelah dropna (customer_name & payment_method):", len(df7))

# Deteksi & Penanganan Duplicate
print("\n\n == Duplicate ===")
print("Baris duplicate (semua kolom sama):", df7.duplicated().sum())
print("transaction_id duplicate:", df7['transaction_id'].duplicated().sum())

df7 = df7.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df7))

#  Standardisasi Teks Kategorikal
for col in ["category", "payment_method", "shipping_city"]:
    df7[col] = df7[col].astype("string").str.strip().str.title()
df7["payment_method"] = df7["payment_method"].replace({"Cod": "COD"})

print("\n\n == Standardisasi Teks ==")
print("Nilai unik category:", df7["category"].unique())
print("Nilai unik payment_method:", df7["payment_method"].unique())

# Koreksi Tipe Data price
def bersihkan_harga7(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
    try:
        return float(x)
    except ValueError:
        return np.nan

df7["price"] = df7["price"].apply(bersihkan_harga7)
print("\n\n == Koreksi price ==")
print(df7["price"].describe())

# Standardisasi Format Tanggal
def parse_tanggal7(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df7["transaction_date"] = df7["transaction_date"].apply(parse_tanggal7).dt.strftime("%Y-%m-%d")
print("\n\n == Standardisasi Tanggal ==")
print(df7["transaction_date"].head())

# Finalisasi Tipe Data
df7["quantity"] = df7["quantity"].astype(int)
df7["price"] = df7["price"].astype(float)
print("\n (SEED=7): Tipe Data Final")
print(df7.dtypes)

#Ekspor Dataset Bersih
jumlah_bersih_7 = len(df7)
df7.to_csv(f"{DIR_SIMPAN}/transaksi_bersih_seed7.csv", index=False)
print("\n\n == Ekspor ==")
print("Dataset bersih tersimpan:", jumlah_bersih_7, "baris")

(SEED=7): Dataset Sintetis
Jumlah baris: 515


 == Missing Value ==
transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              120
dtype: int64

Jumlah baris setelah dropna (customer_name & payment_method): 495


 == Duplicate ===
Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490


 == Standardisasi Teks ==
Nilai unik category: <StringArray>
['Rumah Tangga', 'Olahraga', 'Elektronik', 'Buku', 'Fashion', 'Kesehatan']
Length: 6, dtype: string
Nilai unik payment_method: <StringArray>
['Transfer Bank', 'Kartu Kredit', 'E-Wallet', 'COD']
Length: 4, dtype: string


 == Koreksi price ==
count    4.900000e+02
mean     7.097347e+05
std      1.776507e+06
min      1.500000e+04
25%      5.000000e+04
50%      1.500000e+05
75%      5.000000e+05
max      1.200000e+07
N

Bandingkan Seed = 42 dengan Seed = 7

In [18]:
mentah_42 = len(pd.read_csv(f"{DIR_SIMPAN}/transaksi_mentah.csv"))
bersih_42 = len(pd.read_csv(f"{DIR_SIMPAN}/transaksi_bersih.csv"))

print("=== Perbandingan Jumlah Baris ===")
print(f"SEED=42  -> mentah: {mentah_42} | bersih: {bersih_42}")
print(f"SEED=7   -> mentah: {jumlah_mentah_7} | bersih: {jumlah_bersih_7}")

=== Perbandingan Jumlah Baris ===
SEED=42  -> mentah: 515 | bersih: 490
SEED=7   -> mentah: 515 | bersih: 490


In [19]:
print("5 baris pertama SEED=42:")
print(df[["transaction_id", "customer_name", "product_name"]].head())

print("\n5 baris pertama SEED=7:")
print(df7[["transaction_id", "customer_name", "product_name"]].head())

5 baris pertama SEED=42:
  transaction_id           customer_name product_name
0       TRX00305         Ophelia Hartati    Quo Basic
1       TRX00500      Cut Maya Wijayanti     Delectus
2       TRX00442  Dr. Ridwan Utama, M.Pd    Animi Max
3       TRX00154     Viman Suwarno, S.H.  Consectetur
5       TRX00132      Ir. Ilyas Setiawan  Ducimus Pro

5 baris pertama SEED=7:
  transaction_id              customer_name      product_name
0       TRX00490  Baktiadi Napitupulu, S.H.  Aspernatur Basic
1       TRX00430              Faizah Kusumo    Voluptates Max
2       TRX00083    Drs. Sari Aryani, M.TI.    Deserunt Basic
3       TRX00121        Ilsa Mahendra, S.T.          Quia Pro
5       TRX00148          Cahyanto Agustina     Officia Basic


Jumlah baris sama antara SEED=42 dan SEED=7 (515 mentah, 490 bersih), tapi ini bukan berarti seed tidak berpengaruh. Jumlah baris ditentukan oleh parameter yang bersifat tetap dalam kode N=500, jumlah baris duplicate yang di-hardcode n=15, dan frac missing value yang tetap (0.02/0.03/0.015) dari total baris yang juga selalu 515. Seed hanya mengacak konten dan posisi seperti nama pelanggan, harga, tanggal, kota, serta baris mana yang kena missing value atau diduplikasi berbeda antar seed, meskipun jumlah totalnya konsisten. Ini terbukti dari isi df dan df7 yang berbeda meski len()-nya sama.

## **Latihan 2 - Menambahkan kolom is_valid_price**

In [20]:
# LATIHAN 2: Tambahkan kolom is_valid_price

df["is_valid_price"] = df["price"] > 0

print("Jumlah harga valid vs tidak valid")
print(df["is_valid_price"].value_counts())

print("\nJumlah harga tidak valid (price <= 0 atau NaN):", (~df["is_valid_price"]).sum())

# Melihat baris-baris dengan harga tidak valid
print("\nContoh baris dengan harga tidak valid:")
print(df[~df["is_valid_price"]][["transaction_id", "product_name", "price"]])

Jumlah harga valid vs tidak valid
is_valid_price
True    490
Name: count, dtype: int64

Jumlah harga tidak valid (price <= 0 atau NaN): 0

Contoh baris dengan harga tidak valid:
Empty DataFrame
Columns: [transaction_id, product_name, price]
Index: []


## **Latihan 3 - Jumlah Transaksi per kategori**

In [21]:
jumlah_per_kategori = df["category"].value_counts()
persen_per_kategori = df["category"].value_counts(normalize=True) * 100

ringkasan_kategori = pd.DataFrame({
    "jumlah_transaksi": jumlah_per_kategori,
    "persentase (%)": persen_per_kategori.round(2)
})
print(ringkasan_kategori)

              jumlah_transaksi  persentase (%)
category                                      
Olahraga                    97            19.8
Kesehatan                   91           18.57
Elektronik                  89           18.16
Buku                        82           16.73
Fashion                     66           13.47
Rumah Tangga                65           13.27
